In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import copy
import glob
import random
from random import random as rd
import gc



In [2]:
TRAIN_PATH = "../input/ventilator-pressure-prediction/train.csv"
TEST_PATH = "../input/ventilator-pressure-prediction/test.csv"
SAMPLE_SUB_PATH = "../input/ventilator-pressure-prediction/sample_submission.csv"

if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = "/kaggle/input/ventilator-pressure-prediction/train.csv"
    TEST_PATH = "/kaggle/input/ventilator-pressure-prediction/test.csv"
    SAMPLE_SUB_PATH = (
        "/kaggle/input/ventilator-pressure-prediction/sample_submission.csv"
    )

df_train = pd.read_csv(TRAIN_PATH)

unique_pressures = df_train["pressure"].unique()
sorted_pressures = np.sort(unique_pressures)
total_pressures_len = len(sorted_pressures)


def find_nearest(prediction):
    insert_idx = np.searchsorted(sorted_pressures, prediction)
    if insert_idx == total_pressures_len:
        return float(sorted_pressures[-1])
    elif insert_idx == 0:
        return float(sorted_pressures[0])
    lower_val = sorted_pressures[insert_idx - 1]
    upper_val = sorted_pressures[insert_idx]
    return (
        float(lower_val)
        if abs(lower_val - prediction) < abs(upper_val - prediction)
        else float(upper_val)
    )


def set_seed(seed=2021):
    np.random.seed(seed)
    random_state = np.random.RandomState(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    return random_state


def wc(input_list):
    l = []
    for i in range(len(input_list)):
        base = os.path.basename(input_list[i])
        public_lb_score = 1
        try:
            public_lb_score = int(base.split(".")[1].split(" ")[0])
        except Exception:
            public_lb_score = 1
        l.append(public_lb_score)
        input_list[i] = (pd.read_csv(input_list[i]).pressure).to_numpy().ravel()

    l_sum = sum(l) if sum(l) != 0 else 1
    if len(input_list) == 1:
        output = input_list[0]
    else:
        weight1 = (l[1] / l_sum) + 0.1
        weight2 = 1 - weight1
        output = input_list[0] * weight1 + input_list[1] * weight2
    return output


def g(dp):
    l = []
    for i in glob.iglob(f"{dp}/*"):
        l.append(i)
    file_count = len(l)
    loop_time = 154
    splits = file_count // 2
    l.sort()
    flist = []
    for i in range(splits):
        if i == splits - 1:
            flist.append(l[i * round(len(l) / splits) :])
        else:
            flist.append(
                l[i * round(len(l) / splits) : (i + 1) * round(len(l) / splits)]
            )
    for i in range(len(flist)):
        flist[i] = wc(flist[i])

    pred_list = []
    for k in range(loop_time):
        weight = []
        set_seed(k)
        for j in range(len(flist)):
            weight.append(rd())
        weight_sum = sum(weight) if sum(weight) != 0 else 1.0
        for j in range(len(weight)):
            weight[j] /= weight_sum
        weight.sort(reverse=True)
        temp = 0
        for j in range(len(flist)):
            temp += flist[j] * weight[j]
        pred_list.append(temp)
        del temp
        gc.collect()

    output = pd.read_csv(SAMPLE_SUB_PATH)
    output["pressure"] = np.median(np.vstack(pred_list), axis=0)
    output["pressure"] = output["pressure"].apply(find_nearest)
    output.to_csv(f"rwb {loop_time} loops.csv", index=False)


def blend(a, b, out_path="submission.csv"):
    a_df = pd.read_csv(a)
    b_df = pd.read_csv(b)

    if "id" not in a_df.columns or "pressure" not in a_df.columns:
        raise ValueError(f"File {a} does not have required columns: id, pressure")
    if "id" not in b_df.columns or "pressure" not in b_df.columns:
        raise ValueError(f"File {b} does not have required columns: id, pressure")

    a_df = a_df.sort_values("id").reset_index(drop=True)
    b_df = b_df.sort_values("id").reset_index(drop=True)

    if not np.array_equal(a_df["id"].to_numpy(), b_df["id"].to_numpy()):
        b_df = b_df.set_index("id").reindex(a_df["id"]).reset_index()

    a_df["pressure"] = a_df["pressure"] * 0.6 + b_df["pressure"] * 0.4
    a_df["pressure"] = a_df["pressure"].apply(find_nearest)
    a_df.to_csv(out_path, index=False)
    return a_df




In [3]:
def _discover_candidate_prediction_csvs(search_roots):
    candidates = []
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for p in glob.iglob(os.path.join(root, "**", "*.csv"), recursive=True):
            base = os.path.basename(p).lower()
            if base in {"train.csv", "test.csv", "sample_submission.csv"}:
                continue
            try:
                sz = os.path.getsize(p)
            except OSError:
                continue
            if sz > 100 * 1024 * 1024:  # 100MB guardrail
                continue
            try:
                df = pd.read_csv(p, nrows=5)
            except Exception:
                continue
            if ("id" in df.columns) and ("pressure" in df.columns):
                candidates.append(p)

    def _score_from_name(path):
        base = os.path.basename(path)
        try:
            return float(base.split(" ")[0])
        except Exception:
            try:
                return float(base.split("_")[0])
            except Exception:
                return 999.0

    candidates = sorted(set(candidates), key=_score_from_name)
    return candidates


def _add_breath_features(df):
    # Change (correctness -> better MAE): preserve original row mapping strictly by original index
    # so breath-wise sorted feature engineering cannot detach features from the true `id` rows.
    df = df.copy()
    df["_orig_idx"] = df.index.astype(np.int64)

    # Keep id if present (helps sanity/debug; merge is by id downstream)
    has_id = "id" in df.columns

    df = df.sort_values(["breath_id", "time_step", "_orig_idx"], kind="mergesort")

    g = df.groupby("breath_id", sort=False)

    df["u_in_lag1"] = g["u_in"].shift(1).fillna(0.0)
    df["u_in_lag2"] = g["u_in"].shift(2).fillna(0.0)

    df["u_in_cum"] = g["u_in"].cumsum()
    df["u_in_cum_lag1"] = g["u_in_cum"].shift(1).fillna(0.0)

    df["u_in_diff1"] = (df["u_in"] - df["u_in_lag1"]).astype(np.float64)
    df["u_in_diff1_cum"] = g["u_in_diff1"].cumsum()

    dt_raw = g["time_step"].diff().fillna(0.0).astype(np.float64)
    dt_clip = np.clip(dt_raw.to_numpy(dtype=np.float64), 1e-4, 0.2)
    df["dt"] = dt_clip.astype(np.float64)

    du = df["u_in_diff1"].to_numpy(dtype=np.float64)
    r = np.clip(df["R"].to_numpy(dtype=np.float64), 1.0, None)
    df["du_dt"] = (du / dt_clip).astype(np.float64)
    df["r_du_dt"] = (r * df["du_dt"].to_numpy(dtype=np.float64)).astype(np.float64)

    df["u_in_dt"] = (
        df["u_in"].to_numpy(dtype=np.float64) * df["dt"].to_numpy(dtype=np.float64)
    ).astype(np.float64)
    df["u_in_dt_cum"] = g["u_in_dt"].cumsum().astype(np.float64)

    df["step_idx"] = g.cumcount().astype(np.float64)
    df["u_in_t"] = (
        df["u_in"].to_numpy(dtype=np.float64)
        * df["time_step"].to_numpy(dtype=np.float64)
    ).astype(np.float64)

    df["u_out_lag1"] = g["u_out"].shift(1).fillna(0).astype(np.float64)
    last_uout1_time = df["time_step"].where(df["u_out"] == 1.0)
    last_uout1_time = (
        last_uout1_time.groupby(df["breath_id"], sort=False).ffill().fillna(0.0)
    )
    df["time_since_uout1"] = (df["time_step"] - last_uout1_time).astype(np.float64)

    c = np.clip(df["C"].to_numpy(dtype=np.float64), 1.0, None)
    r2 = np.clip(df["R"].to_numpy(dtype=np.float64), 1.0, None)

    df["x0"] = df["u_in"].to_numpy(dtype=np.float64) / c
    df["x1"] = df["u_in_cum"].to_numpy(dtype=np.float64) / c
    df["x2"] = df["u_in_lag1"].to_numpy(dtype=np.float64) / c
    df["x3"] = df["u_in_lag2"].to_numpy(dtype=np.float64) / c
    df["x4"] = df["u_in_cum_lag1"].to_numpy(dtype=np.float64) / c
    df["x5"] = df["u_in"].to_numpy(dtype=np.float64) / (c * np.sqrt(r2))
    df["x6"] = df["time_step"].to_numpy(dtype=np.float64)
    df["x7"] = (df["u_in"].to_numpy(dtype=np.float64) == 0.0).astype(np.float64)

    df["x8"] = df["u_in_diff1_cum"].to_numpy(dtype=np.float64) / c
    df["x9"] = df["u_in_dt_cum"].to_numpy(dtype=np.float64) / c

    df["x10"] = df["step_idx"].to_numpy(dtype=np.float64) / 80.0  # normalize ~[0,1]
    df["x11"] = df["u_in_t"].to_numpy(dtype=np.float64) / c

    df["x12"] = df["dt"].to_numpy(dtype=np.float64)

    df["x13"] = df["u_out_lag1"].to_numpy(dtype=np.float64)
    df["x14"] = df["time_since_uout1"].to_numpy(dtype=np.float64)

    df["x15"] = (df["r_du_dt"].to_numpy(dtype=np.float64) / 1000.0).astype(np.float64)

    # Restore exact incoming row order by the saved original index (no reliance on id ordering)
    df = df.sort_values("_orig_idx", kind="mergesort").drop(columns=["_orig_idx"])
    return df


def _robust_fit_linear(X, y):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    Xd = np.hstack([np.ones((X.shape[0], 1), dtype=np.float64), X])

    beta, *_ = np.linalg.lstsq(Xd, y, rcond=None)
    resid = y - Xd @ beta

    mad = np.median(np.abs(resid - np.median(resid))) + 1e-6
    s = 1.4826 * mad
    k = 1.345 * s
    w = 1.0 / np.maximum(1.0, np.abs(resid) / k)  # in (0,1]
    W = w[:, None]

    beta2, *_ = np.linalg.lstsq(Xd * W, y * w, rcond=None)
    return beta2  # includes intercept as beta2[0]


search_roots = [
    "../input",
    "/kaggle/input",
    "../kaggle/input",
]
cands = _discover_candidate_prediction_csvs(search_roots)

sample = pd.read_csv(SAMPLE_SUB_PATH)
sample = sample.sort_values("id").reset_index(drop=True)

test_df = pd.read_csv(
    TEST_PATH, usecols=["id", "breath_id", "time_step", "u_in", "u_out", "C", "R"]
)

if len(cands) >= 2:
    sub = blend(cands[0], cands[1], out_path="submission.csv")
elif len(cands) == 1:
    one = pd.read_csv(cands[0]).sort_values("id").reset_index(drop=True)
    if not np.array_equal(one["id"].to_numpy(), sample["id"].to_numpy()):
        one = one.set_index("id").reindex(sample["id"]).reset_index()
    one["pressure"] = one["pressure"].apply(find_nearest)
    one[["id", "pressure"]].to_csv("submission.csv", index=False)
    sub = one[["id", "pressure"]]
else:
    train_cols = ["breath_id", "time_step", "u_in", "u_out", "C", "R", "pressure"]
    tr = df_train[train_cols].copy()
    tr = _add_breath_features(tr)

    te = test_df.copy()
    te = _add_breath_features(te)

    feat_cols = [
        "x0",
        "x1",
        "x2",
        "x3",
        "x4",
        "x5",
        "x6",
        "x7",
        "x8",
        "x9",
        "x10",
        "x11",
        "x12",
        "x13",
        "x14",
        "x15",
    ]

    insp_fit = tr.loc[(tr["u_out"] == 0)].copy()

    params = []
    for (r, c), gdf in insp_fit.groupby(["R", "C"], sort=False):
        X = gdf[feat_cols].to_numpy(dtype=np.float64)
        y = gdf["pressure"].to_numpy(dtype=np.float64)
        beta = _robust_fit_linear(X, y)
        params.append((int(r), int(c)) + tuple(beta.tolist()))
    params_df = pd.DataFrame(
        params,
        columns=["R", "C"] + [f"b{i}" for i in range(0, len(feat_cols) + 1)],
    )

    Xg = insp_fit[feat_cols].to_numpy(dtype=np.float64)
    yg = insp_fit["pressure"].to_numpy(dtype=np.float64)
    beta_g = _robust_fit_linear(Xg, yg)

    exp = tr.loc[tr["u_out"] == 1, ["R", "C", "pressure"]]
    exp_med = (
        exp.groupby(["R", "C"], sort=False)["pressure"]
        .median()
        .reset_index()
        .rename(columns={"pressure": "exp_pressure"})
    )
    exp_global = float(df_train.loc[df_train["u_out"] == 1, "pressure"].median())

    insp0 = tr.loc[(tr["u_out"] == 0) & (tr["u_in"] == 0), ["R", "C", "pressure"]]
    insp0_med = (
        insp0.groupby(["R", "C"], sort=False)["pressure"]
        .median()
        .reset_index()
        .rename(columns={"pressure": "insp0_pressure"})
    )
    insp0_global = float(
        df_train.loc[
            (df_train["u_out"] == 0) & (df_train["u_in"] == 0), "pressure"
        ].median()
    )

    merged = sample.merge(test_df, on="id", how="left")
    merged = merged.merge(te[["id"] + feat_cols], on="id", how="left")
    merged = merged.merge(params_df, on=["R", "C"], how="left")
    merged = merged.merge(exp_med, on=["R", "C"], how="left")
    merged = merged.merge(insp0_med, on=["R", "C"], how="left")

    beta_cols = [f"b{i}" for i in range(0, len(feat_cols) + 1)]
    for k, v in zip(beta_cols, beta_g.tolist()):
        merged[k] = merged[k].fillna(float(v))
    merged["exp_pressure"] = merged["exp_pressure"].fillna(exp_global)
    merged["insp0_pressure"] = merged["insp0_pressure"].fillna(insp0_global)

    X_test = merged[feat_cols].to_numpy(dtype=np.float64)
    b0 = merged["b0"].to_numpy(dtype=np.float64)
    B = merged[[f"b{i}" for i in range(1, len(feat_cols) + 1)]].to_numpy(
        dtype=np.float64
    )
    pred_insp = b0 + np.sum(B * X_test, axis=1)

    u_out_arr = merged["u_out"].to_numpy(dtype=np.int64)
    u_in_arr = merged["u_in"].to_numpy(dtype=np.float64)

    pred = pred_insp.copy()
    pred = np.where(
        u_out_arr == 1, merged["exp_pressure"].to_numpy(dtype=np.float64), pred
    )
    pred = np.where(
        (u_out_arr == 0) & (u_in_arr == 0),
        merged["insp0_pressure"].to_numpy(dtype=np.float64),
        pred,
    )

    merged["pressure"] = pred
    merged["pressure"] = merged["pressure"].apply(find_nearest)
    merged[["id", "pressure"]].to_csv("submission.csv", index=False)
    sub = merged[["id", "pressure"]]

print("Wrote submission.csv with shape:", sub.shape)
print(sub.head())

Wrote submission.csv with shape: (603600, 2)
   id   pressure
0   1   8.298067
1   2   8.930786
2   3   9.141693
3   4  10.688340
4   5  12.445893
